# MobileNetV2 — Find visually similar images

Use case: image-based search in a catalog. We demonstrate the retrieval mechanism with CIFAR-10 images. MobileNetV2 is a lightweight CNN. An embedding is a numeric vector summarizing image features.

Run the cells from top to bottom. Each notebook is independent.


## Step 1 — Install packages
Use Python 3.11 or 3.12 as a straightforward training environment, or Google Colab. Install into the active notebook kernel. Restart the kernel after the first installation if required. Internet is needed initially for packages, sample images/data and ImageNet weights. A GPU is optional; CPU runs can be slower. Model downloads can be hundreds of MB.


In [ ]:
%pip install tensorflow numpy matplotlib scikit-learn pillow


## Step 2 — Import libraries
TensorFlow runs the neural network. Keras provides the model API. NumPy handles arrays and Matplotlib displays pictures.


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras

keras.utils.set_random_seed(42)
print("TensorFlow:", tf.__version__)


## Step 3 — Load the demonstration catalog
CIFAR-10 provides small labeled images. Labels are used only to help inspect results, not to calculate similarity.


In [ ]:
(train_images, train_labels), (test_images, test_labels) = keras.datasets.cifar10.load_data()
train_labels = train_labels.ravel()
test_labels = test_labels.ravel()
class_names = ["airplane", "automobile", "bird", "cat", "deer",
               "dog", "frog", "horse", "ship", "truck"]
print("Training images:", train_images.shape)
print("Test images:", test_images.shape)
plt.figure(figsize=(9, 4))
for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(train_images[i])
    plt.title(class_names[train_labels[i]])
    plt.axis("off")
plt.tight_layout()
plt.show()


## Step 4 — Build a feature extractor
include_top=False removes the original classifier. Global average pooling produces one fixed-length vector per image.


In [ ]:
from tensorflow.keras.applications.mobilenet_v2 import MobileNetV2, preprocess_input
extractor = MobileNetV2(weights="imagenet", include_top=False,
                        pooling="avg", input_shape=(224, 224, 3))
catalog = train_images[:300]
catalog_labels = train_labels[:300]


## Step 5 — Extract vectors in small batches
Upsampling makes the input compatible but does not restore missing detail. Small batches limit memory use. L2 normalization makes cosine similarity a dot product.


In [ ]:
feature_batches = []
for start in range(0, len(catalog), 16):
    batch = tf.image.resize(catalog[start:start + 16], (224, 224))
    features = extractor.predict(preprocess_input(batch), verbose=0)
    feature_batches.append(features)
embeddings = np.concatenate(feature_batches)
embeddings = embeddings / np.maximum(np.linalg.norm(embeddings, axis=1, keepdims=True), 1e-8)
print("Embedding shape:", embeddings.shape)


## Step 6 — Search for nearest images
Exclude the query itself so that the first result is not a trivial exact match. Higher cosine similarity indicates closer vectors.


In [ ]:
query_index = 42
scores = embeddings @ embeddings[query_index]
scores[query_index] = -np.inf
nearest = np.argsort(scores)[::-1][:5]
plt.figure(figsize=(13, 3))
plt.subplot(1, 6, 1)
plt.imshow(catalog[query_index])
plt.title("Query")
plt.axis("off")
for position, index in enumerate(nearest, start=2):
    plt.subplot(1, 6, position)
    plt.imshow(catalog[index])
    plt.title(f"{class_names[catalog_labels[index]]}\n{scores[index]:.3f}")
    plt.axis("off")
plt.tight_layout()
plt.show()


## Understand the result
Similarity is not classification accuracy. The closest picture may have different content. For a real product catalog, use clear product photos and evaluate retrieval against relevant-item judgments.


## Practice
1. Change the input image or selected image index.
2. Explain the output in your own words.
3. Compare the result with the original image and record one limitation.


## References
- [Keras Applications](https://keras.io/api/applications/)
- [CIFAR-10 dataset](https://www.cs.toronto.edu/~kriz/cifar.html)
- [Keras Grad-CAM example](https://keras.io/examples/vision/grad_cam/)
